## $p$-median Problem 

### B2B Supply Hub / Commissary Location for Davao City Coffee & Matcha Shops
The $p$-median B2B supply hub Facility Location Problem (FLP) aims to locate exactly $p$ micro-warehouses within Davao City to minimize the total demand-weighted travel distance between all 400 retail coffee/matcha shops and their assigned supply hub. In this optimization model, estimated inventory order volumes act as node weights ($w_i$), and the primary objective is to maximize spatial logistics efficiency while operating under a fixed number of $p$ facilities to open.

1. **Objective Function**: Minimize the demand-weighted distance:
$$\text{minimize} \; Z = \sum_{j \in J}\sum_{i \in I} w_i d_{ij} y_{ij}$$ 
2. **Constraints**
* Ensures that each demand node (cafe) is assigned to exactly one facility:
$$\sum_{j \in J} y_{ij} = 1 \quad \forall i \in I$$ 
* Ensures that a cafe is not assigned to a candidate warehouse that is not opened ($y_{ij} = 1 \Rightarrow x_j = 1$):
$$y_{ij} \leq x_j \quad \forall i \in I, j \in J$$
* Ensures that there are exactly $p$ facilities opened:
$$\sum_{j \in J} x_j = p$$
* Binary Integrality Constraints:
$$x_j, y_{ij} \in \{1,0\} \quad \forall i \in I, j \in J$$
3. **Decision Variables**
* $x_j \in \{1,0\}$: 1 if the distribution hub is opened at candidate location $j$, and 0 otherwise. 
* $y_{ij} \in \{1,0\}$: 1 if demand node (cafe) $i$ is assigned to facility $j$, and 0 otherwise.
4. **Constants & Parameters**
* $w_i$: the weight or inventory demand at cafe node $i$
* $d_{ij}$: the distance (in meters) between demand node $i$ and candidate facility $j$
* $p$: the number of distribution hubs to be located


Either install these packages on Jupyter or in your terminal if not yet installed:
```bash
%pip install docplex haversine ipywidgets plotly
```

In [ ]:
from docplex.mp.model import Model
import numpy as np
import pandas as pd
from scipy.spatial.distance import cdist
from haversine import haversine
import plotly.express as px
import plotly.graph_objects as go

#### Step 1: Data Preprocessing & Synthetic Parameter Generation (Optional if CSVs are already generated)
This cell documents how `davao_coffee_matcha_nodes.csv` and `candidate_warehouses.csv` are prepared from `davao_coffee_matcha_shops.csv`.

In [ ]:
# 1. Load original Davao Coffee & Matcha Shops dataset and generate synthetic demand weights
shops_df = pd.read_csv("davao_coffee_matcha_shops.csv", encoding="utf-8")
np.random.seed(42)

# Lognormal base demand + spatial proximity multiplier centered on Poblacion (7.075, 125.605)
base_demand = np.random.lognormal(mean=4.5, sigma=0.45, size=len(shops_df))
dist_from_center = np.sqrt((shops_df["latitude"] - 7.075)**2 + (shops_df["longitude"] - 125.605)**2)
proximity_factor = np.clip(1.3 - (dist_from_center * 3), 0.7, 1.3)

shops_df["demand_weight"] = np.round(base_demand * proximity_factor).astype(int)
shops_df["ID"] = range(len(shops_df))
shops_df["LE_ENGLISH"] = shops_df["place_name"]
shops_df["POPULATION"] = shops_df["demand_weight"]
shops_df["Y"] = shops_df["latitude"]
shops_df["X"] = shops_df["longitude"]

nodes_cols = ["ID", "place_name", "latitude", "longitude", "demand_weight", "LE_ENGLISH", "X", "Y", "POPULATION"]
shops_df = shops_df[nodes_cols]
shops_df.to_csv("davao_coffee_matcha_nodes.csv", index=False)

# 2. Generate 52 Candidate Warehouse Locations across Davao City corridors
warehouses_data = [
    (0, "Sta. Ana Port Logistics Hub", "Leon Garcia", "Poblacion", 7.0782, 125.6235, 65000, 1800),
    (1, "Monteverde Commercial Depot", "Tomas Monteverde", "Poblacion", 7.0754, 125.6188, 60000, 1500),
    (2, "Uyanguren Wholesale Commissary", "Barangay 27-C", "Poblacion", 7.0768, 125.6199, 62000, 1600),
    (3, "Quirino Central Cold Storage", "Barangay 3-A", "Poblacion", 7.0691, 125.6052, 58000, 1400),
    (4, "Claveria Roastery & Supply Center", "Barangay 32-D", "Poblacion", 7.0722, 125.6124, 61000, 1500),
    (5, "Obrero Micro-Fulfillment Center", "Barangay 18-B", "Poblacion", 7.0858, 125.6145, 59000, 1500),
    (6, "Obrero North Dispatch Hub", "Barangay 19-B", "Poblacion", 7.0912, 125.6118, 57000, 1400),
    (7, "Bankerohan Agro-Supply Depot", "Barangay 5-A", "Poblacion", 7.0678, 125.6001, 55000, 1600),
    (8, "Boulevard Coastal Warehouse", "Barangay 23-C", "Poblacion", 7.0685, 125.6192, 52000, 1300),
    (9, "Rizal Park Commercial Storage", "Barangay 34-D", "Poblacion", 7.0652, 125.6095, 58000, 1350),
    (10, "Bajada Victoria Logistics Park", "Barangay 20-B", "Poblacion", 7.0955, 125.6132, 64000, 1750),
    (11, "Cabaguio Industrial Complex", "Paciano Bangoy", "Agdao", 7.0928, 125.6221, 56000, 1650),
    (12, "Agdao Public Market Supply Hub", "Agdao Proper", "Agdao", 7.0841, 125.6248, 54000, 1550),
    (13, "R. Castillo Portside Warehouse", "Wilfredo Aquino", "Agdao", 7.0984, 125.6312, 58000, 1800),
    (14, "Damosa IT & Logistics Center", "Angliongto", "Buhangin", 7.1042, 125.6285, 68000, 2000),
    (15, "Lanang Business Park Commissary", "Vicente Hizon Sr.", "Buhangin", 7.1105, 125.6392, 66000, 1900),
    (16, "Pampanga Cargo & Cold Chain", "Pampanga", "Buhangin", 7.1168, 125.6475, 63000, 1850),
    (17, "Sasa Wharf Freight Terminal", "Sasa", "Buhangin", 7.1274, 125.6562, 70000, 2400),
    (18, "Panacan Industrial Zone A", "Panacan", "Bunawan", 7.1520, 125.6558, 65000, 2200),
    (19, "Tibungco North Supply Hub", "Tibungco", "Bunawan", 7.1895, 125.6521, 50000, 1800),
    (20, "Bunawan Highway Logistics", "Bunawan Proper", "Bunawan", 7.2345, 125.6465, 45000, 1700),
    (21, "Buhangin Gym Commercial Hub", "Buhangin Proper", "Buhangin", 7.1132, 125.6125, 55000, 1600),
    (22, "Milan Diversion Logistics", "Buhangin Proper", "Buhangin", 7.1205, 125.6184, 59000, 1750),
    (23, "Cabantian Commercial Depot", "Cabantian", "Buhangin", 7.1415, 125.6152, 48000, 1500),
    (24, "Communal Airport Cargo Hub", "Communal", "Buhangin", 7.1352, 125.6341, 62000, 1900),
    (25, "Indangan North Micro-Hub", "Indangan", "Buhangin", 7.1585, 125.6082, 44000, 1300),
    (26, "Mandug Agro-Distribution Center", "Mandug", "Buhangin", 7.1542, 125.5824, 42000, 1400),
    (27, "Maa Diversion Freight Center", "Ma-a", "Talomo", 7.0875, 125.5912, 60000, 1850),
    (28, "Maa-Magtuod Junction Warehouse", "Ma-a", "Talomo", 7.0762, 125.5815, 53000, 1600),
    (29, "Langub Hilltop Logistics", "Langub", "Talomo", 7.0892, 125.5684, 46000, 1400),
    (30, "Bucana Coastal Supply Depot", "Bucana", "Talomo", 7.0545, 125.6082, 57000, 1550),
    (31, "Ecoland Terminal Dispatch Hub", "Bucana", "Talomo", 7.0528, 125.5992, 63000, 1900),
    (32, "Matina Town Square Micro-Hub", "Matina Crossing", "Talomo", 7.0615, 125.5934, 59000, 1650),
    (33, "Matina Aplaya Storage Facility", "Matina Aplaya", "Talomo", 7.0435, 125.5882, 52000, 1500),
    (34, "Matina Pangi Diversion Depot", "Matina Pangi", "Talomo", 7.0715, 125.5672, 51000, 1600),
    (35, "Bangkal McArthur Logistics", "Talomo Proper", "Talomo", 7.0568, 125.5695, 56000, 1750),
    (36, "Ulas Junction Distribution Hub", "Talomo Proper", "Talomo", 7.0485, 125.5482, 58000, 1850),
    (37, "Catalunan Grande Supply Center", "Catalunan Grande", "Talomo", 7.0812, 125.5395, 49000, 1500),
    (38, "Catalunan Pequeno Commissary", "Catalunan Pequeno", "Talomo", 7.0642, 125.5245, 47000, 1450),
    (39, "Baliok Coastal Logistics", "Baliok", "Talomo", 7.0345, 125.5122, 46000, 1400),
    (40, "Dumoy Cold Chain & Dairy Hub", "Dumoy", "Talomo", 7.0265, 125.5185, 54000, 1800),
    (41, "Bago Aplaya South Depot", "Bago Aplaya", "Talomo", 7.0382, 125.5324, 51000, 1600),
    (42, "Puan Highway Fulfillment Center", "Talomo Proper", "Talomo", 7.0425, 125.5418, 53000, 1650),
    (43, "Toril Poblacion Logistics Center", "Toril Proper", "Toril", 7.0185, 125.4982, 55000, 1900),
    (44, "Lizada Industrial Park Hub", "Lizada", "Toril", 7.0055, 125.4895, 50000, 1800),
    (45, "Mulig South-West Roastery", "Mulig", "Toril", 7.0312, 125.4785, 44000, 1500),
    (46, "Mintal Commercial Supply Hub", "Mintal", "Tugbok", 7.0895, 125.5042, 50000, 1650),
    (47, "Tugbok District Warehouse", "Tugbok Proper", "Tugbok", 7.0982, 125.5185, 47000, 1550),
    (48, "Bago Oshiro Agro-Processing Hub", "Bago Oshiro", "Tugbok", 7.0715, 125.4985, 46000, 1500),
    (49, "Los Amigos Highway Depot", "Los Amigos", "Tugbok", 7.1245, 125.4752, 43000, 1400),
    (50, "Calinan Highland Coffee Hub", "Calinan Proper", "Calinan", 7.1885, 125.4525, 48000, 1800),
    (51, "Buda Mountain Supply Outpost", "Marilog Proper", "Marilog", 7.3050, 125.2850, 38000, 1200)
]

columns = ["id", "service_name", "barangay", "district", "y", "x", "fixed_cost_php", "capacity_kg"]
warehouses_df = pd.DataFrame(warehouses_data, columns=columns)
warehouses_df["latitude"] = warehouses_df["y"]
warehouses_df["longitude"] = warehouses_df["x"]
warehouses_df.to_csv("candidate_warehouses.csv", index=False)
print("Datasets generated and saved successfully.")

#### Define the model name
Create a `Model` object named `davao_b2b_flp`.

In [ ]:
mdl = Model(name="davao_b2b_flp")

#### Read the CSV files containing the input data.
Read the CSV file containing the demand node input data (`davao_coffee_matcha_nodes.csv`).

In [ ]:
nodes_df = pd.read_csv("davao_coffee_matcha_nodes.csv", encoding="utf-8")
nodes_df.head()

Read the CSV file containing the candidate facilities input data (`candidate_warehouses.csv`).

In [ ]:
facilities_df = pd.read_csv("candidate_warehouses.csv", encoding="utf-8")
facilities_df.head()

#### Define Constants and Parameters 
Extract the weight/demand $w_i$ at node $i$. Set $p$ (number of hubs to open) to 8.

In [ ]:
nodes = nodes_df["ID"].tolist()
facilities = facilities_df["id"].tolist()
names = dict(zip(nodes_df["ID"], nodes_df["LE_ENGLISH"]))

nodes_df['POPULATION'] = nodes_df['POPULATION'].astype(str).str.replace(',', '', regex=False)
nodes_df['POPULATION'] = pd.to_numeric(nodes_df['POPULATION'], errors='coerce')
w = dict(zip(nodes_df["ID"], nodes_df["POPULATION"]))

p = 8

Compute the distance matrix $d_{ij}$ between the demand nodes and the candidate facilities using `cdist` from `scipy.spatial.distance` with `metric=haversine`. Coordinates are passed in `(latitude, longitude)` decimal degrees (`Y`, `X`), and distances are converted from kilometers to integer meters.

In [ ]:
node_coords = nodes_df.loc[:, ["Y", "X"]]
facility_coords = facilities_df.loc[:, ["y", "x"]]

# Haversine returns distance in kilometers; multiply by 1000 to convert to integer meters
d = cdist(node_coords, facility_coords, metric=haversine)
d = np.round(d * 1000, decimals=0).astype(int)

#### Define the Decision Variables
Binary variables $x_j$ for the candidate facilities.

In [ ]:
x = mdl.binary_var_dict(facilities, lb=0, name="x_%s")

Binary variables $y_{ij}$ for the node-facility assignment.

In [ ]:
y = mdl.binary_var_matrix(keys1=nodes, keys2=facilities, name="y_%s_%s")

#### Define the Constraints
Node Assignment Constraints: $\sum_{j \in J} y_{ij} = 1 \quad \forall i \in I$

In [ ]:
mdl.add_constraints(mdl.sum(y[i, j] for j in facilities) == 1 for i in nodes);

Open Facility Assignment Constraints: $y_{ij} \leq x_j \quad \forall i \in I, j \in J$

In [ ]:
mdl.add_constraints(y[i, j] <= x[j]
    for i in nodes
    for j in facilities);

Number of Facilities Constraint: $\sum_{j \in J} x_j = p$

In [ ]:
mdl.add_constraint(mdl.sum(x[j] for j in facilities) == p);

#### Define the Objective Function
Minimize total demand-weighted distance: $\sum_{j \in J}\sum_{i \in I} w_i d_{ij} y_{ij}$

In [ ]:
mdl.minimize(mdl.sum(w[i] * d[i, j] * y[i, j] for i in nodes for j in facilities))

#### Solve the Model and Retrieve the Solution
The following helper functions retrieve the optimal values of $x_j$ and $y_{ij}$, store them into DataFrames, and plot the Davao City hub-to-cafe assignments on an interactive map.

In [ ]:
def retrieve_solution(mdl, facilities_df, nodes_df):
    solution_df = mdl.solution.as_df()
    y_df = solution_df.loc[solution_df['name'].str.contains("y")]
    x_df = solution_df.loc[solution_df['name'].str.contains("x")]
    x_df = x_df['name'].str.split('_', n=2, expand=True)
    x_df.drop(0, axis=1, inplace=True)

    # Retrieve the facilities that are selected
    inc_fac_idx = [int(x) for x in list(x_df[1])]
    inc_fac_coords = []
    for index in inc_fac_idx:
        inc_fac_coords.append((facilities_df.iloc[int(index)]['y'], facilities_df.loc[int(index)]['x']))
    inc_fac_df = pd.DataFrame(inc_fac_coords, columns=['lat', 'long'])
    inc_fac_df['id'] = [int(x) for x in inc_fac_idx]
    inc_fac_df = pd.merge(inc_fac_df['id'], facilities_df, on='id')

    # Retrieve the assigned demand nodes
    y_df = y_df[~(y_df['value'] <= 0.5)]  
    y_df = y_df['name'].str.split('_', n=3, expand=True)
    y_df.drop(0, axis=1, inplace=True)
    y_df = y_df.rename(columns={1: 'ID', 2: 'facility_id'}).reset_index()
    y_df.drop("index", axis=1, inplace=True)
    y_df['ID'] = y_df['ID'].astype('int64')
    y_df['facility_id'] = y_df['facility_id'].astype('int')
    y_df = pd.merge(y_df, nodes_df, on='ID')

    return inc_fac_df, y_df

In [ ]:
def plot_assignments(inc_fac_df, y_df): 
    fig = px.scatter_map(
        inc_fac_df, lat="y", lon="x", hover_name="service_name",
        zoom=10.5, center=dict(lat=7.085, lon=125.58), height=850
    )
    fig.update_traces(marker_size=15, marker_color='rgb(0,0,255)')
    fig.update_layout(mapbox_style="carto-positron", margin={"r": 0, "t": 0, "l": 0, "b": 0})

    for i in set(inc_fac_df["id"]):
        lats = []
        longs = []
        nodes_assigned = y_df.loc[y_df['facility_id'] == i]
        f_lat = inc_fac_df[inc_fac_df["id"] == i]['y'].iloc[0]
        f_long = inc_fac_df[inc_fac_df["id"] == i]['x'].iloc[0]
        for j in range(len(nodes_assigned)):
            n_lat = nodes_assigned.iloc[j]['Y']
            n_long = nodes_assigned.iloc[j]['X']
            lats.append(n_lat)
            longs.append(n_long)
            lats.append(f_lat)
            longs.append(f_long)
        
        fig.add_trace(go.Scattermap(
            mode="lines", lon=longs, lat=lats, 
            line={'width': 2, 'color': 'rgb(0,0,0)'}, showlegend=False
        ))

    fig.add_trace(go.Scattermap(
        mode="markers", lat=y_df['Y'].tolist(), lon=y_df['X'].tolist(),
        text=y_df['LE_ENGLISH'].tolist(),  
        marker={'size': 8, 'color': 'rgb(255,0,0)'}, showlegend=False
    ))

    fig.show()

In [ ]:
print("Solving model...")
mdl.parameters.timelimit = 3600  # 1-hour time limit per project guidelines
solution = mdl.solve(log_output=True)

if solution:
    print(f"Solution Status: {mdl.solve_details.status}")
    print(f"Computational Time: {mdl.solve_details.time:.4f} sec.")
    print(f"Objective Value (Total Weighted Distance in kg-meters): {mdl.objective_value:,.2f}")
    
    inc_fac_df, y_df = retrieve_solution(mdl, facilities_df, nodes_df)
    
    # Summarize hub assignments
    summary = y_df.groupby('facility_id').agg(
        cafes_assigned=('ID', 'count'),
        total_demand_kg=('POPULATION', 'sum')
    ).reset_index().merge(inc_fac_df[['id', 'service_name', 'barangay', 'district']], left_on='facility_id', right_on='id')
    display(summary[['facility_id', 'service_name', 'barangay', 'district', 'cafes_assigned', 'total_demand_kg']])
    
    plot_assignments(inc_fac_df, y_df)
else:
    print("No solution found.")